# NZEB Climate-Aware Transfer Learning
This notebook runs the experiments for the research paper *Transfer Learning for Cross-Climate Building Energy Prediction: Investigating Data Efficiency and Generalization*.


## Phase 0: Setup
Ensure the `src` directory is accessible (either uploaded to Google Drive or in the same directory as this notebook).


In [ ]:
import sys
import os

# Determine Environment and set SRC_PATH
IN_COLAB = False
IN_KAGGLE = False
SRC_PATH = None

if 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    IN_KAGGLE = True
    print('Running on Kaggle!')
    # On Kaggle, datasets are usually in /kaggle/input and working dir in /kaggle/working
    # We search for the 'src' directory
    for root, dirs, files in os.walk('/kaggle'):
        if 'src' in dirs:
            try:
                if 'config.py' in os.listdir(os.path.join(root, 'src')):
                    SRC_PATH = root
                    break
            except:
                pass
    
    if not SRC_PATH:
        print('❌ src folder not found in /kaggle. Please upload the src folder as a Kaggle Dataset.')
else:
    try:
        import google.colab
        from google.colab import drive
        drive.mount('/content/drive', force_remount=True)
        IN_COLAB = True
        print('Running on Colab!')
        
        base_drive = '/content/drive/MyDrive'
        for folder in os.listdir(base_drive):
            potential_path = os.path.join(base_drive, folder)
            try:
                if os.path.isdir(potential_path) and 'src' in os.listdir(potential_path):
                    SRC_PATH = potential_path
                    break
            except:
                continue
    except (ImportError, NotImplementedError):
        print('Running locally!')
        pass

if not SRC_PATH:
    # Fallback to local search
    curr_dir = os.getcwd()
    if 'src' in os.listdir(curr_dir):
        SRC_PATH = curr_dir
    elif 'src' in os.listdir(os.path.dirname(curr_dir)):
        SRC_PATH = os.path.dirname(curr_dir)

if SRC_PATH:
    print(f'✅ Found project root: {SRC_PATH}')
    if SRC_PATH not in sys.path:
        sys.path.append(SRC_PATH)
    if not IN_KAGGLE:
        os.chdir(SRC_PATH)
    else:
        os.chdir('/kaggle/working')
else:
    print('WARNING: Could not automatically find the project root folder (containing src/).')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from src import config, data_loader, feature_engineering, metrics, model, transfer, visualization

print('Setup complete!')


## Phase 1: Data Loading & Exploration
Load pre-merged dataset from Drive and validate.


In [ ]:
# The data is already in your mounted Google Drive shortcut or Kaggle input!
csv_path = 'merged_ashrae.csv'
if IN_KAGGLE:
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'merged_ashrae.csv' in files:
            csv_path = os.path.join(root, 'merged_ashrae.csv')
            print(f'Found dataset at {csv_path}')
            break

try:
    df = data_loader.load_merged_data(csv_path)
    # Filter to electricity only
    df = df[df['meter'] == 0].copy()
    data_loader.validate_data(df, expected_sites=list(config.SITES.keys()))
    
    site_dfs = data_loader.split_by_site(df)
    print("Available sites:", site_dfs.keys())
    
    # Exploration
    for site_id, site_df in site_dfs.items():
        print(f"\nSite {site_id} ({config.SITES[site_id]}):")
        print(f"Shape: {site_df.shape}")
        
except FileNotFoundError:
    print(f"Please place your merged data at {csv_path} before continuing.")


## Phase 2: Feature Engineering
Apply temporal features, log transform, and scaling.


In [ ]:
# Process features for all sites
processed_dfs = {}
all_uses = set()

# Get all primary uses across all sites for consistent encoding
if 'df' in locals():
    if 'primary_use' in df.columns:
        all_uses = set(df['primary_use'].unique())

X_dict = {}
y_dict = {}
features_list = None

if 'site_dfs' in locals():
    for site_id, site_df in site_dfs.items():
        X, y, f_names = feature_engineering.prepare_features(site_df, all_possible_uses=all_uses)
        X_dict[site_id] = X
        y_dict[site_id] = y
        features_list = f_names # Same for all due to all_possible_uses
        
    print("Features ready:", len(features_list), "features.")



## Phase 3: Baseline Models (5-Fold CV)
Train scratch models for each site to establish baseline performance.


In [ ]:
baseline_results = []

if 'X_dict' in locals():
    for site_id in config.SITES.keys():
        print(f"\nTraining Baseline for Site {site_id} ({config.SITES[site_id]})...")
        X_site = X_dict[site_id]
        y_site = y_dict[site_id]
        
        X_scaled, _, _ = feature_engineering.scale_features(X_site, X_site)
        
        results = model.train_with_kfold(
            X_scaled, y_site, 
            n_folds=config.N_FOLDS,
            layers=config.ANN_LAYERS,
            learning_rate=config.LEARNING_RATE,
            epochs=config.EPOCHS,
            patience=config.PATIENCE,
            site_name=config.SITES[site_id]
        )
        
        summary = metrics.summarize_kfold(results)
        summary['site_id'] = site_id
        summary['location'] = config.SITES[site_id]
        summary['model_type'] = 'Scratch'
        baseline_results.append(summary)
        
    df_baselines = pd.DataFrame(baseline_results)
    print(df_baselines[['location', 'R2_mean', 'R2_std', 'MAPE_mean']])


## Phase 4: Transfer Learning (Secondary Scenarios)
Perform transfer learning for the secondary scenarios using 5-fold CV.


In [ ]:
transfer_results = []

if 'X_dict' in locals():
    for scenario in config.SECONDARY_SCENARIOS:
        src = scenario['source']
        tgt = scenario['target']
        print(f"\nSecondary Transfer: {config.SITES[src]} -> {config.SITES[tgt]}")
        
        model_name = f"{config.SITES[src]}_baseline_fold_1.keras"
        source_model_path = f"/kaggle/working/{model_name}"
        if not os.path.exists(source_model_path):
            source_model_path = model_name
            if IN_KAGGLE:
                for root, _, files in os.walk('/kaggle/input'):
                    if model_name in files:
                        source_model_path = os.path.join(root, model_name)
                        break
            
        if os.path.exists(source_model_path):
            X_tgt_scaled, _, _ = feature_engineering.scale_features(X_dict[tgt], X_dict[tgt])
            
            t_res = transfer.transfer_with_kfold(
                source_model_path,
                X_tgt_scaled, y_dict[tgt],
                n_folds=config.N_FOLDS,
                n_freeze=config.N_FREEZE_LAYERS,
                lr=config.TRANSFER_LEARNING_RATE,
                epochs=config.EPOCHS,
                patience=config.PATIENCE,
                site_name=f"{config.SITES[src]}_to_{config.SITES[tgt]}"
            )
            summary = metrics.summarize_kfold(t_res)
            summary['source'] = config.SITES[src]
            summary['target'] = config.SITES[tgt]
            summary['model_type'] = 'Transfer'
            summary['climate_dist'] = config.CLIMATE_DISTANCE_MATRIX[src][tgt]
            transfer_results.append(summary)
        else:
            print(f"Source model not found: {source_model_path}")
            
    if transfer_results:
        df_transfer = pd.DataFrame(transfer_results)
        print(df_transfer[['source', 'target', 'climate_dist', 'R2_mean', 'R2_std']])


## Phase 5: Primary Experiment - Data Efficiency
Evaluate transfer vs scratch on varying fractions of target data for the primary scenario (Tempe -> Cornell).


In [ ]:
efficiency_results = []

print("Running Phase 5 Data Efficiency loop...")
src = config.PRIMARY_SCENARIO['source']
tgt = config.PRIMARY_SCENARIO['target']
X_tgt_full = X_dict[tgt]
y_tgt_full = y_dict[tgt]

model_name = f"{config.SITES[src]}_baseline_fold_1.keras"
source_model_path = f"/kaggle/working/{model_name}"
if not os.path.exists(source_model_path):
    source_model_path = model_name
    if IN_KAGGLE:
        for root, _, files in os.walk('/kaggle/input'):
            if model_name in files:
                source_model_path = os.path.join(root, model_name)
                break

import random

for fraction in config.DATA_FRACTIONS:
    print(f"\n--- Evaluating Target Data Fraction: {fraction*100:.0f}% ---")
    for seed in config.SEEDS:
        np.random.seed(seed)
        random.seed(seed)
        
        n_samples = max(10, int(len(X_tgt_full) * fraction))
        indices = np.random.choice(len(X_tgt_full), n_samples, replace=False)
        X_sample = X_tgt_full.iloc[indices] if hasattr(X_tgt_full, 'iloc') else X_tgt_full[indices]
        y_sample = y_tgt_full.iloc[indices] if hasattr(y_tgt_full, 'iloc') else y_tgt_full[indices]
        
        X_scaled, _, _ = feature_engineering.scale_features(X_sample, X_sample)
        
        scratch_model = model.build_ann(X_scaled.shape[1], layers=config.ANN_LAYERS)
        scratch_model = model.compile_model(scratch_model, learning_rate=config.LEARNING_RATE)
        from tensorflow import keras
        es_cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=config.PATIENCE, restore_best_weights=True)
        h_scratch = scratch_model.fit(X_scaled, y_sample, epochs=config.EPOCHS, batch_size=config.BATCH_SIZE, validation_split=0.2, callbacks=[es_cb], verbose=1)
        
        y_pred_scratch = scratch_model.predict(X_scaled, verbose=0).flatten()
        res_scratch = metrics.evaluate_original_scale(y_sample, y_pred_scratch)
        res_scratch['Method'] = 'Scratch'
        res_scratch['Data_Fraction'] = fraction
        res_scratch['Seed'] = seed
        res_scratch['Accuracy_Proxy'] = 100 - (res_scratch['MAPE'] * 100)
        efficiency_results.append(res_scratch)
        
        if os.path.exists(source_model_path):
            transfer_model = transfer.load_source_model(source_model_path)
            transfer_model = transfer.freeze_layers(transfer_model, n_freeze=config.N_FREEZE_LAYERS)
            transfer_model = transfer.recompile_for_transfer(transfer_model, learning_rate=config.TRANSFER_LEARNING_RATE)
            from tensorflow import keras
            es_cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=config.PATIENCE, restore_best_weights=True)
            h_transfer = transfer_model.fit(X_scaled, y_sample, epochs=config.EPOCHS, batch_size=config.BATCH_SIZE, validation_split=0.2, callbacks=[es_cb], verbose=1)
            
            y_pred_transfer = transfer_model.predict(X_scaled, verbose=0).flatten()
            res_transfer = metrics.evaluate_original_scale(y_sample, y_pred_transfer)
            res_transfer['Method'] = 'Transfer'
            res_transfer['Data_Fraction'] = fraction
            res_transfer['Seed'] = seed
            res_transfer['Accuracy_Proxy'] = 100 - (res_transfer['MAPE'] * 100)
            efficiency_results.append(res_transfer)

df_eff = pd.DataFrame(efficiency_results)
df_eff.to_csv("data_efficiency_results.csv", index=False)
print("Data efficiency results saved to data_efficiency_results.csv")

df_agg = df_eff.groupby(['Data_Fraction', 'Method']).mean(numeric_only=True).reset_index()
df_agg.rename(columns={'R2': 'R2_mean', 'Method': 'model_type'}, inplace=True)
try:
    scratch_100 = df_agg[(df_agg['Data_Fraction'] == 1.0) & (df_agg['model_type'] == 'Scratch')]['R2_mean'].values[0]
except IndexError:
    scratch_100 = 0.0

fig = visualization.plot_data_efficiency_primary(df_agg, scratch_100, save_name="Primary_Data_Efficiency")
plt.show()

if 'h_scratch' in locals() and 'h_transfer' in locals():
    fig2 = visualization.plot_convergence_comparison(h_transfer, h_scratch, fraction, save_name=f"Convergence_Plots")
    plt.show()


## Phase 6: Climate Analysis
Domain justification using climate distance.


In [ ]:
print("Phase 6: Climate Analysis (Heatmap)")
import numpy as np

sites_idx = [2, 4, 15]
labels = [config.SITES[s] for s in sites_idx]
matrix = np.zeros((3, 3))

for i, sid in enumerate(sites_idx):
    if 'df_baselines' in locals():
        row = df_baselines[df_baselines['site_id'] == sid]
        if not row.empty:
            matrix[i, i] = row['R2_mean'].values[0]

if 'df_transfer' in locals():
    for _, row in df_transfer.iterrows():
        src_label = row['source']
        tgt_label = row['target']
        try:
            s_idx = labels.index(src_label)
            t_idx = labels.index(tgt_label)
            matrix[s_idx, t_idx] = row['R2_mean']
        except ValueError:
            pass

fig = visualization.plot_transferability_heatmap(matrix, labels, labels, save_name="Transferability_Heatmap")
plt.show()


## Phase 7: Explainability (SHAP)
Identify which features transfer.


In [ ]:
print("Phase 7: SHAP Explainability")
if 'transfer_model' in locals() and 'X_scaled' in locals():
    try:
        import shap
        bg_sample = X_scaled[:100]
        explainer = shap.DeepExplainer(transfer_model, bg_sample)
        
        test_sample = X_scaled[100:150]
        shap_values = explainer.shap_values(test_sample)
        
        if isinstance(shap_values, list):
            shap_values = shap_values[0]
            
        plt.figure(figsize=(10, 8))
        shap.summary_plot(shap_values, features=test_sample, feature_names=features_list, show=False)
        plt.title("SHAP Feature Importance (Post-Transfer)")
        plt.tight_layout()
        if not os.path.exists("figures"):
            os.makedirs("figures")
        plt.savefig("figures/SHAP_Summary.png", dpi=300)
        plt.show()
        print("SHAP plot saved.")
    except Exception as e:
        print(f"SHAP explainer failed (often version compatibility with Keras 3): {e}")
else:
    print("No transfer model available to explain.")
